In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error,classification_report


In [ ]:
df = pd.read_csv('/home/revanth/projects/spotify-analysis-ml/data/spotify-tracks-dataset/dataset.csv')
df['duration_min'] = df['duration_ms'] / 60000
df = pd.get_dummies(df, columns=['track_genre'])
df['explicit'] = df['explicit'].astype(int)
df = pd.get_dummies(df, columns=['key'])

In [3]:
df.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='str')

In [4]:
# Select ONLY the columns that contain integers or decimals
numeric_df = df.select_dtypes(include=['int64', 'float64', 'bool']).copy()

columns_to_drop = [col for col in ['Unnamed: 0', 'duration_ms'] if col in numeric_df.columns]
if columns_to_drop:
    numeric_df = numeric_df.drop(columns=columns_to_drop)



In [5]:
numeric_df.head()

,popularity,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,73,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4
1,55,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4
2,57,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4
3,71,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3
4,82,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4


In [6]:

X1 = numeric_df.drop(columns=['popularity'])
y1 = numeric_df['popularity'] # Real 0-100 scale

X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y1, test_size=0.2, random_state=42)

model1 = LinearRegression()
model1.fit(X_train1, y_train1)
predictions1 = model1.predict(X_test1)

print("--- EXPERIMENT 1: WITH ZEROS (0-100 SCALE) ---")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test1, predictions1):.2f} points")
print(f"R-squared Score (R2):       {r2_score(y_test1, predictions1):.2f}")
print(f"Loss (RMSE):               {root_mean_squared_error(y_test1, predictions1):.2f} points\n")

--- EXPERIMENT 1: WITH ZEROS (0-100 SCALE) ---
Mean Absolute Error (MAE): 18.34 points
R-squared Score (R2):       0.02
Loss (RMSE):               21.96 points



In [7]:
df_non_zero = numeric_df[numeric_df['popularity'] > 0].copy()

X2 = df_non_zero.drop(columns=['popularity']) 
y2 = df_non_zero['popularity'] # Safe matching row counts on a 1-100 scale

X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X_train2, y_train2)
predictions2 = model2.predict(X_test2)

print("--- EXPERIMENT 2: WITHOUT ZEROS (1-100 SCALE) ---")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test2, predictions2):.2f} points")
print(f"R-squared Score (R2):       {r2_score(y_test2, predictions2):.2f}")
print(f"Loss (RMSE):               {root_mean_squared_error(y_test2, predictions2):.2f} points")

# %%
# Check for Overfitting on Experiment 2
train_preds2 = model2.predict(X_train2)
test_preds2 = model2.predict(X_test2)
print("--- OVERFITTING CHECK (EXPERIMENT 2) ---")
print(f"Train RMSE: {root_mean_squared_error(y_train2, train_preds2):.2f} | Test RMSE: {root_mean_squared_error(y_test2, test_preds2):.2f}")

--- EXPERIMENT 2: WITHOUT ZEROS (1-100 SCALE) ---
Mean Absolute Error (MAE): 15.28 points
R-squared Score (R2):       0.06
Loss (RMSE):               18.72 points
--- OVERFITTING CHECK (EXPERIMENT 2) ---
Train RMSE: 18.53 | Test RMSE: 18.72


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X3 = df_non_zero.drop(columns=['popularity'])
median_popularity = df_non_zero['popularity'].median()

y3 = (df_non_zero['popularity'] > median_popularity).astype(int)

print("--- LOGISTIC REGRESSION SETUP ---")
print(f"Popularity Median Cutoff Point: {median_popularity}")
print(f"Class Distribution:\n{y3.value_counts()}\n")

X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y3, test_size=0.2, random_state=42)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train3, y_train3)

predictions3 = log_model.predict(X_test3)

accuracy = accuracy_score(y_test3, predictions3)

print("--- CLASSIFICATION PERFORMANCE ---")
print(f"Overall Model Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Classification Report:")
print(classification_report(y_test3, predictions3))

--- LOGISTIC REGRESSION SETUP ---
Popularity Median Cutoff Point: 39.0
Class Distribution:
popularity
0    49393
1    48587
Name: count, dtype: int64

--- CLASSIFICATION PERFORMANCE ---
Overall Model Accuracy: 59.52%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.58      0.59      9833
           1       0.59      0.61      0.60      9763

    accuracy                           0.60     19596
   macro avg       0.60      0.60      0.60     19596
weighted avg       0.60      0.60      0.60     19596



In [9]:
coef_df = pd.DataFrame({
    "feature": X_train2.columns,
    "coefficient": model2.coef_
})

coef_df.sort_values(by="coefficient", ascending=False)

,feature,coefficient
1,danceability,7.132373
0,explicit,4.546053
12,time_signature,0.667968
4,loudness,0.241586
11,tempo,-0.008331
3,key,-0.009708
5,mode,-0.631304
7,acousticness,-1.126793
9,liveness,-2.328515
10,valence,-7.065649
